# Chat with PDF

This notebook demonstrates the full Chat with PDF pipeline:
1. Extract text from a PDF using PyPDF2
2. Chunk the text into overlapping segments
3. Generate embeddings using OpenAI
4. Store embeddings in ChromaDB
5. Ask questions and retrieve answers using RAG (Retrieval-Augmented Generation)

In [24]:
from IPython.display import Video
Video("./video/demo.mov", height=500)

![title](./image/IMG_0079.PNG)

## 1. Install Dependencies

In [22]:
import sys
!{sys.executable} -m pip install chromadb openai PyPDF2

## 2. Configuration

In [12]:
import os
import getpass

# Set your OpenAI API key here or via environment variable
#OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-openai-api-key-here")
OPENAI_API_KEY =  getpass.getpass("Paste your OpenAI API key: ")
    
# Path to the PDF file to process
PDF_PATH = "./pdf/ai_engineer_cv.pdf"

# Chunking parameters
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Retrieval parameters
TOP_K = 5

print("Configuration loaded.")
print(f"PDF path: {PDF_PATH}")
print(f"OpenAI API key set: {OPENAI_API_KEY != 'your-openai-api-key-here'}")

Paste your OpenAI API key:  ········


Configuration loaded.
PDF path: ./pdf/ai_engineer_cv.pdf
OpenAI API key set: True


## 3. Import Libraries

In [13]:
import uuid
from pathlib import Path
from typing import List

import openai
import chromadb
from chromadb.config import Settings
from PyPDF2 import PdfReader

print("All libraries imported successfully.")

All libraries imported successfully.


## 4. Initialize Clients

In [14]:
# Initialize OpenAI client
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client initialized.")

# Initialize ChromaDB client (in-memory for notebook use)
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
print("ChromaDB client initialized (in-memory).")

OpenAI client initialized.
ChromaDB client initialized (in-memory).


## 5. Step 1 — Extract Text from PDF

In [15]:
def extract_text_from_pdf(file_path: str) -> str:
    """Extract text from all pages of a PDF."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


pdf_text = extract_text_from_pdf(PDF_PATH)

print(f"Total characters extracted: {len(pdf_text):,}")
print(f"\n--- First 500 characters ---\n")
print(pdf_text[:500])

Total characters extracted: 854

--- First 500 characters ---

AI Engineer - CV
 
Name: John Doe
Email: john.doe@email.com | Phone: +33 6 00 00 00 00
Location: Paris, France
Professional Summary
AI Engineer with experience in machine learning, deep learning, and scalable AI systems. Skilled in building
end-to-end ML pipelines, deploying models, and working with LLM-based applications.
Skills
Python, PyTorch, TensorFlow, Scikit-learn, NLP, LLMs, LangChain, FastAPI, Docker, SQL, Cloud (AWS/GCP)
Experience
AI Engineer - XYZ Tech (2023 - Present)
Developed and 


## 6. Step 2 — Chunk the Text

In [16]:
def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
    """Split text into overlapping chunks, breaking at sentence boundaries."""
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]

        # Try to break at a sentence boundary
        if end < text_length:
            last_period = chunk.rfind('.')
            last_newline = chunk.rfind('\n')
            break_point = max(last_period, last_newline)
            if break_point > chunk_size * 0.5:
                chunk = chunk[:break_point + 1]
                end = start + break_point + 1

        chunks.append(chunk.strip())
        start = end - overlap

    return [c for c in chunks if c]  # Remove empty chunks


chunks = chunk_text(pdf_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) // len(chunks)} characters")
print(f"\n--- Chunk 0 (first 300 chars) ---\n")
print(chunks[0][:300])
print(f"\n--- Chunk 1 (first 300 chars) ---\n")
print(chunks[1][:300] if len(chunks) > 1 else "(only one chunk)")

Total chunks created: 2
Average chunk length: 452 characters

--- Chunk 0 (first 300 chars) ---

AI Engineer - CV
 
Name: John Doe
Email: john.doe@email.com | Phone: +33 6 00 00 00 00
Location: Paris, France
Professional Summary
AI Engineer with experience in machine learning, deep learning, and scalable AI systems. Skilled in building
end-to-end ML pipelines, deploying models, and working with

--- Chunk 1 (first 300 chars) ---

MSc in Artificial Intelligence - University of Paris


## 7. Step 3 — Generate Embeddings with OpenAI

In [17]:
def get_embeddings(texts: List[str]) -> List[List[float]]:
    """Generate embeddings for a list of texts using OpenAI."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]


print(f"Generating embeddings for {len(chunks)} chunks...")
embeddings = get_embeddings(chunks)

print(f"Embeddings generated: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")

Generating embeddings for 2 chunks...
Embeddings generated: 2
Embedding dimensions: 1536


## 8. Step 4 — Store in ChromaDB

In [18]:
# Generate a unique document ID
document_id = str(uuid.uuid4())[:8]
collection_name = f"doc_{document_id}"

# Create the collection with cosine similarity
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

# Add all chunks with their embeddings and metadata
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
    metadatas=[
        {"chunk_index": i, "document_id": document_id}
        for i in range(len(chunks))
    ]
)

print(f"Document ID: {document_id}")
print(f"Collection: {collection_name}")
print(f"Chunks stored: {collection.count()}")

Document ID: f4b0fc37
Collection: doc_f4b0fc37
Chunks stored: 2


## 9. Step 5 — Ask a Question (RAG)

In [19]:
def ask_question(question: str, top_k: int = TOP_K) -> dict:
    """
    Retrieve relevant chunks and generate an answer using GPT.
    Returns a dict with 'answer' and 'sources'.
    """
    # Embed the question
    question_embedding = get_embeddings([question])[0]

    # Query ChromaDB for the most relevant chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
        include=["documents", "distances", "metadatas"]
    )

    context_chunks = results["documents"][0]
    distances = results["distances"][0]

    # Build context string
    context = "\n\n---\n\n".join(context_chunks)

    system_prompt = """You are a helpful assistant that answers questions based on the provided context from a PDF document.

Rules:
- Only use information from the provided context to answer
- If the context doesn't contain enough information, say so
- Be concise but thorough
- Quote relevant parts when appropriate
- If asked about something not in the context, explain that the information is not available in the document"""

    user_prompt = f"""Context from the document:
{context}

Question: {question}

Please provide a helpful answer based on the context above."""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_tokens=1000
    )

    answer = response.choices[0].message.content

    sources = [
        {
            "text": chunk[:200] + "..." if len(chunk) > 200 else chunk,
            "relevance": round(1 - dist, 3)
        }
        for chunk, dist in zip(context_chunks, distances)
    ]

    return {"answer": answer, "sources": sources}


print("ask_question() function ready.")

ask_question() function ready.


## 10. Chat — Ask Your Questions

In [20]:
question = "What are the main skills and technologies mentioned?"

result = ask_question(question)

print(f"Question: {question}\n")
print(f"Answer:\n{result['answer']}")
print(f"\n--- Top Sources ---")
for i, src in enumerate(result["sources"], 1):
    print(f"\n[{i}] Relevance: {src['relevance']}")
    print(f"    {src['text']}")

Question: What are the main skills and technologies mentioned?

Answer:
The main skills and technologies mentioned in the document are:

- **Programming Languages and Frameworks**: Python, PyTorch, TensorFlow, Scikit-learn
- **Natural Language Processing**: NLP, LLMs (Large Language Models), LangChain
- **Web Development**: FastAPI
- **Containerization**: Docker
- **Databases**: SQL
- **Cloud Platforms**: AWS (Amazon Web Services), GCP (Google Cloud Platform)

These skills highlight the candidate's expertise in machine learning, deep learning, and scalable AI systems.

--- Top Sources ---

[1] Relevance: 0.402
    AI Engineer - CV
 
Name: John Doe
Email: john.doe@email.com | Phone: +33 6 00 00 00 00
Location: Paris, France
Professional Summary
AI Engineer with experience in machine learning, deep learning, and ...

[2] Relevance: 0.245
    MSc in Artificial Intelligence - University of Paris


In [21]:
question = "What is the work experience described in the document?"

result = ask_question(question)

print(f"Question: {question}\n")
print(f"Answer:\n{result['answer']}")
print(f"\n--- Top Sources ---")
for i, src in enumerate(result["sources"], 1):
    print(f"\n[{i}] Relevance: {src['relevance']}")
    print(f"    {src['text']}")

Question: What is the work experience described in the document?

Answer:
The work experience described in the document includes:

1. **AI Engineer at XYZ Tech (2023 - Present)**:
   - Developed and deployed machine learning models for prediction systems.
   - Built LLM-powered chatbot applications.
   - Optimized inference pipelines for production use.

2. **Machine Learning Intern at ABC Labs (2022 - 2023)**:
   - Worked on data preprocessing.
   - Involved in model training and evaluation for classification and regression problems.

--- Top Sources ---

[1] Relevance: 0.363
    AI Engineer - CV
 
Name: John Doe
Email: john.doe@email.com | Phone: +33 6 00 00 00 00
Location: Paris, France
Professional Summary
AI Engineer with experience in machine learning, deep learning, and ...

[2] Relevance: 0.203
    MSc in Artificial Intelligence - University of Paris


In [ ]:
# Ask your own question
question = "What is the educational background mentioned?"

result = ask_question(question)

print(f"Question: {question}\n")
print(f"Answer:\n{result['answer']}")
print(f"\n--- Top Sources ---")
for i, src in enumerate(result["sources"], 1):
    print(f"\n[{i}] Relevance: {src['relevance']}")
    print(f"    {src['text']}")

## 11. Interactive Chat Loop

In [ ]:
# Run this cell and type your questions interactively. Type 'exit' to stop.

print("Chat with PDF — Interactive Mode")
print("Type 'exit' to quit\n")

while True:
    question = input("You: ").strip()
    if not question or question.lower() == "exit":
        print("Goodbye!")
        break

    result = ask_question(question)
    print(f"\nAssistant: {result['answer']}\n")
    print(f"(Based on {len(result['sources'])} retrieved chunks, top relevance: {result['sources'][0]['relevance']})\n")
    print("-" * 60)

## 12. Cleanup

In [ ]:
# Delete the ChromaDB collection when done
chroma_client.delete_collection(collection_name)
print(f"Collection '{collection_name}' deleted.")